# 69 — P10.8: inventario global y bloqueo de reentrenamiento

No entrena, no abre tests sellados y no deserializa checkpoints.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, re
import pandas as pd

ROOT=Path(os.getenv("PFI_ROOT","/content/drive/MyDrive/PFI_MVP"))
OUT=Path(os.getenv("PFI_P10_8_PREFLIGHT_ROOT",str(ROOT/"results/P10_8_clinical_expansion_preflight")))
REPOS=[Path(p) for p in [
    os.getenv("PFI_AI_REPO",""),
    "/content/PFI_MVPTest_Enzo_AImodule",
    str(ROOT/"PFI_MVPTest_Enzo_AImodule"),
    str(ROOT/"repos/PFI_MVPTest_Enzo_AImodule"),
] if p]
EXPECTED={
 "p10_6_subarticular":"d41262d57b13c146a48ab15f5e183cc6a55fc92724b7d0c286cea1f2ce26e84a",
 "p10_7_best_candidate":"9e22ed3d4ea4150ebbd28875027d7ffd922bbe182aa463fb9e64724d2b363a71",
 "p10_7_frozen_export":"16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293",
}
TASKS=[
 ("central_canal_stenosis","Estenosis central","P10.6","severity_multiclass"),
 ("neural_foraminal_narrowing_left","Estrechamiento foraminal izquierdo","P10.6","severity_multiclass"),
 ("neural_foraminal_narrowing_right","Estrechamiento foraminal derecho","P10.6","severity_multiclass"),
 ("subarticular_stenosis_left","Estenosis subarticular izquierda","P10.6","severity_multiclass"),
 ("subarticular_stenosis_right","Estenosis subarticular derecha","P10.6","severity_multiclass"),
 ("pfirrmann_grade","Pfirrmann I–V","P10.7","ordinal_multiclass"),
 ("modic_change","Cambios Modic","P10.7","multiclass"),
 ("upper_endplate_change","Cambio de platillo superior / Schmorl","P10.7","binary"),
 ("lower_endplate_change","Cambio de platillo inferior / Schmorl","P10.7","binary"),
 ("spondylolisthesis","Espondilolistesis binaria","P10.7","binary"),
 ("disc_herniation","Hernia discal binaria","P10.7","binary"),
 ("disc_narrowing","Estrechamiento discal","P10.7","binary"),
 ("disc_bulging","Abombamiento discal","P10.7","binary"),
]
blocked={x[0] for x in TASKS}
requested={x.strip() for x in re.split(r"[,;\n]+",os.getenv("PFI_P10_8_REQUESTED_TASKS","")) if x.strip()}
collision=sorted(requested & blocked)
if collision:
    raise RuntimeError("RETRAINING_GUARD_BLOCKED: "+", ".join(collision))

def sha(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

def roots(paths):
    seen=set(); result=[]
    for p in paths:
        if p.exists():
            key=str(p.resolve())
            if key not in seen: seen.add(key); result.append(p)
    return result

def write_json(path,payload):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(payload,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    os.replace(tmp,path)

scan=roots([ROOT/"models",ROOT/"results",*REPOS])
ck=[]
seen=set()
for base in scan:
    for p in base.rglob("*.pt"):
        k=str(p.resolve())
        if k in seen: continue
        seen.add(k); digest=sha(p)
        ck.append({"path":str(p),"fileName":p.name,"sizeBytes":p.stat().st_size,
                   "sha256":digest,"expectedArtifactMatches":[n for n,v in EXPECTED.items() if v==digest],
                   "loadedWithTorch":False})

nbs=[]
seen=set()
for base in roots(REPOS):
    for p in base.rglob("*.ipynb"):
        k=str(p.resolve())
        if k in seen: continue
        seen.add(k)
        nbs.append({"repoRoot":str(base),"relativePath":str(p.relative_to(base)),
                    "fileName":p.name,"sizeBytes":p.stat().st_size,"sha256":sha(p)})

ev=[]
patterns=["*COMPLETE*.json","*complete*.json","*summary*.json","*manifest*.json","*MODEL_CARD*.md","*.modelcard.md"]
seen=set()
for base in scan:
    for pattern in patterns:
        for p in base.rglob(pattern):
            k=str(p.resolve())
            if p.is_dir() or k in seen: continue
            seen.add(k); ev.append({"path":str(p),"fileName":p.name,"sizeBytes":p.stat().st_size,"sha256":sha(p)})

task_df=pd.DataFrame([{"taskId":a,"displayName":b,"projectPhase":c,"outputType":d,
                       "alreadyTrained":True,"blockedForRetraining":True,
                       "productClaim":"hallazgo candidato para revisión profesional; no diagnóstico autónomo"}
                      for a,b,c,d in TASKS])
ck_df=pd.DataFrame(ck)
nb_df=pd.DataFrame(nbs)
ev_df=pd.DataFrame(ev)

OUT.mkdir(parents=True,exist_ok=True)
task_df.to_csv(OUT/"task_registry_v1.csv",index=False)
nb_df.to_csv(OUT/"notebook_registry_v1.csv",index=False)
ev_df.to_csv(OUT/"evidence_registry_v1.csv",index=False)
write_json(OUT/"checkpoint_registry_v1.json",{
    "schemaVersion":"pfi.p10-8.checkpoint-registry.v1","expectedHashes":EXPECTED,
    "weightsDeserialized":False,"checkpoints":ck})
dup_name=(nb_df.groupby("fileName").size().loc[lambda s:s>1].to_dict() if not nb_df.empty else {})
dup_sha=(nb_df.groupby("sha256").size().loc[lambda s:s>1].to_dict() if not nb_df.empty else {})
write_json(OUT/"notebook_duplicate_audit_v1.json",{
    "schemaVersion":"pfi.p10-8.notebook-duplicate-audit.v1",
    "duplicateNames":dup_name,"duplicateHashes":dup_sha})
marker={
 "schemaVersion":"pfi.p10-8.notebook-69-complete.v1","status":"NOTEBOOK_69_COMPLETE",
 "generatedAtUtc":datetime.now(timezone.utc).isoformat(),"trainingExecuted":False,
 "weightsDeserialized":False,"internalTestAccessed":False,"officialHiddenTestAccessed":False,
 "patientIdentifiersExported":False,"blockedTaskCount":len(TASKS),
 "checkpointCount":len(ck_df),"notebookCount":len(nb_df),"evidenceFileCount":len(ev_df),
 "requestedTasks":sorted(requested),"blockedRequestedTasks":collision}
write_json(OUT/"NOTEBOOK_69_COMPLETE.json",marker)
print(json.dumps(marker,indent=2,ensure_ascii=False))
print("NOTEBOOK_69_COMPLETE")